In [5]:
import os
import json
import time
from pydantic import BaseModel, Field
from typing import Optional

"""
=============================================================================
COMPREHENSIVE GUIDE: Prompt Caching with Instructor + OpenAI
=============================================================================

This guide covers EVERY caching technique available when using the
`instructor` library with OpenAI. Each example shows:
  - The technique and when to use it
  - Full working code
  - How to read cost/usage metrics (tokens, cached tokens, cost)
  - Prompt structure tips that maximize cache hits

SETUP:
    pip install instructor openai pydantic diskcache redis tiktoken

Set your API key:
    export OPENAI_API_KEY="sk-..."

=============================================================================
TABLE OF CONTENTS
=============================================================================
1. Understanding OpenAI's Automatic Prompt Caching (server-side)
2. Tracking Usage & Cached Tokens with Instructor
3. Technique 1: Long System Prompt — Static Prefix Pattern
4. Technique 2: Tool/Function Definitions Caching
5. Technique 3: Structured Output Schema Caching
6. Technique 4: Multi-turn Conversation Caching
7. Technique 5: Large Context Document Caching
8. Technique 6: prompt_cache_key for Routing Hints
9. Technique 7: Instructor's Built-in Client-Side Cache (AutoCache)
10. Technique 8: functools.cache — In-Memory Client-Side Cache
11. Technique 9: diskcache — Persistent Client-Side Cache
12. Technique 10: Redis — Distributed Client-Side Cache
13. Cost Calculator Utility
14. Best Practices Cheat Sheet
"""

# ============================================================================
# 0. SETUP & IMPORTS
# ============================================================================

In [47]:
import instructor
from openai import OpenAI

# Initialize the raw OpenAI client (for some examples)
raw_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Initialize the Instructor-patched client
client = instructor.from_provider("openai/gpt-4.1", temperature=0.0, top_p=1.0)

# ============================================================================
# UTILITY: Print usage metrics from any OpenAI completion object
# ============================================================================

In [7]:
# Pricing per 1M tokens (as of early 2025 — check OpenAI pricing page)
PRICING = {
    "gpt-4o": {"input": 2.50, "cached_input": 1.25, "output": 10.00},
    "gpt-4o-mini": {"input": 0.15, "cached_input": 0.075, "output": 0.60},
    "gpt-4o-2024-08-06": {"input": 2.50, "cached_input": 1.25, "output": 10.00},
    "gpt-4.1-mini": {"input": 0.40, "cached_input": 0.10, "output": 1.60},
    "gpt-4.1": {"input": 2.00, "cached_input": 0.50, "output": 8.00},
    "o1-preview": {"input": 15.00, "cached_input": 7.50, "output": 60.00},
    "o1-mini": {"input": 3.00, "cached_input": 1.50, "output": 12.00},
}

In [8]:
def print_usage(completion, label="", model="gpt-4o-mini"):
    """
    Prints detailed usage metrics from an OpenAI completion object.
    Works with both raw OpenAI responses and Instructor's create_with_completion.
    """
    usage = completion.usage
    prompt_tokens = usage.prompt_tokens
    completion_tokens = usage.completion_tokens
    total_tokens = usage.total_tokens

    # Extract cached tokens (may be nested differently depending on SDK version)
    cached_tokens = 0
    if hasattr(usage, "prompt_tokens_details") and usage.prompt_tokens_details:
        cached_tokens = getattr(usage.prompt_tokens_details, "cached_tokens", 0) or 0

    uncached_tokens = prompt_tokens - cached_tokens
    cache_hit_rate = (cached_tokens / prompt_tokens * 100) if prompt_tokens > 0 else 0

    # Calculate cost
    pricing = PRICING.get(model, PRICING["gpt-4o-mini"])
    input_cost = (uncached_tokens / 1_000_000) * pricing["input"]
    cached_cost = (cached_tokens / 1_000_000) * pricing["cached_input"]
    output_cost = (completion_tokens / 1_000_000) * pricing["output"]
    total_cost = input_cost + cached_cost + output_cost

    # What it WOULD have cost without caching
    full_input_cost = (prompt_tokens / 1_000_000) * pricing["input"]
    full_cost = full_input_cost + output_cost
    savings = full_cost - total_cost

    print(f"\n{'='*60}")
    print(f"  📊 USAGE METRICS {f'— {label}' if label else ''}")
    print(f"{'='*60}")
    print(f"  Prompt tokens:      {prompt_tokens:>8,}")
    print(f"    ├─ Cached:        {cached_tokens:>8,}  ({'✅' if cached_tokens > 0 else '❌'})")
    print(f"    └─ Uncached:      {uncached_tokens:>8,}")
    print(f"  Completion tokens:  {completion_tokens:>8,}")
    print(f"  Total tokens:       {total_tokens:>8,}")
    print(f"  Cache hit rate:     {cache_hit_rate:>7.1f}%")
    print(f"{'─'*60}")
    print(f"  💰 COST BREAKDOWN (model: {model})")
    print(f"  Input (uncached):   ${input_cost:.6f}")
    print(f"  Input (cached):     ${cached_cost:.6f}")
    print(f"  Output:             ${output_cost:.6f}")
    print(f"  TOTAL:              ${total_cost:.6f}")
    print(f"  Savings vs no-cache:${savings:.6f}")
    print(f"{'='*60}\n")

    return {
        "prompt_tokens": prompt_tokens,
        "cached_tokens": cached_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
        "cache_hit_rate": cache_hit_rate,
        "total_cost": total_cost,
        "savings": savings,
    }

# ============================================================================
# 1. UNDERSTANDING OPENAI'S AUTOMATIC PROMPT CACHING (SERVER-SIDE)
# ============================================================================
"""

In [9]:
"""
KEY FACTS about OpenAI's server-side prompt caching:

✅ AUTOMATIC — No code changes needed. OpenAI caches for you.
✅ FREE — No extra charge for cache storage. Cached tokens cost 50% less
           (or up to 90% less on newer models like gpt-4.1).
✅ PREFIX-BASED — Caching matches the LONGEST prefix of your prompt that
                  was previously seen, starting at 1,024 tokens.
✅ GRANULARITY — Cached in 128-token increments after the first 1,024.
✅ TTL — Cache stays warm for 5-10 minutes of inactivity (default).
         24-hour retention available for gpt-4.1 and gpt-5.1 series.
✅ SCOPE — Caches are per-organization, not shared across orgs.

WHAT GETS CACHED (the entire request prefix):
  - System messages
  - User/assistant message history
  - Tool/function definitions
  - Structured output schemas
  - Images and audio (if identical)

GOLDEN RULE FOR CACHE HITS:
  ┌─────────────────────────────────────────────┐
  │  PUT STATIC CONTENT FIRST (system prompt,   │
  │  tool defs, schemas, examples, documents)   │
  │                                             │
  │  PUT DYNAMIC CONTENT LAST (user query,      │
  │  variable data, per-request context)        │
  └─────────────────────────────────────────────┘

  The prefix must match EXACTLY — even a single token difference
  at position 500 means tokens 501+ can't be cached.
"""

"\nKEY FACTS about OpenAI's server-side prompt caching:\n\n✅ AUTOMATIC — No code changes needed. OpenAI caches for you.\n✅ FREE — No extra charge for cache storage. Cached tokens cost 50% less\n           (or up to 90% less on newer models like gpt-4.1).\n✅ PREFIX-BASED — Caching matches the LONGEST prefix of your prompt that\n                  was previously seen, starting at 1,024 tokens.\n✅ GRANULARITY — Cached in 128-token increments after the first 1,024.\n✅ TTL — Cache stays warm for 5-10 minutes of inactivity (default).\n         24-hour retention available for gpt-4.1 and gpt-5.1 series.\n✅ SCOPE — Caches are per-organization, not shared across orgs.\n\nWHAT GETS CACHED (the entire request prefix):\n  - System messages\n  - User/assistant message history\n  - Tool/function definitions\n  - Structured output schemas\n  - Images and audio (if identical)\n\nGOLDEN RULE FOR CACHE HITS:\n  ┌─────────────────────────────────────────────┐\n  │  PUT STATIC CONTENT FIRST (system prompt,

# ============================================================================
# 2. TRACKING USAGE & CACHED TOKENS WITH INSTRUCTOR
# ============================================================================


In [10]:
"""
Instructor provides `create_with_completion()` which returns BOTH the
parsed Pydantic model AND the raw completion object with usage stats.
"""

'\nInstructor provides `create_with_completion()` which returns BOTH the\nparsed Pydantic model AND the raw completion object with usage stats.\n'

In [11]:
class UserInfo(BaseModel):
    name: str
    age: int


In [13]:
def example_track_usage():
    """Shows how to get usage metrics from every Instructor call."""

    user, completion = client.create_with_completion(
        response_model=UserInfo,
        messages=[
            {"role": "user", "content": "Extract: Jason is 25 years old"},
        ],
    )

    print(f"Extracted: {user}")
    print(f"\nRaw usage object: {completion.usage}")
    print(f"Prompt tokens: {completion.usage.prompt_tokens}")
    print(f"Completion tokens: {completion.usage.completion_tokens}")

    # Access cached tokens
    if completion.usage.prompt_tokens_details:
        print(f"Cached tokens: {completion.usage.prompt_tokens_details.cached_tokens}")

    # Use our utility for a prettier view
    print_usage(completion, label="Basic Usage Tracking")

example_track_usage()

Extracted: name='Jason' age=25

Raw usage object: CompletionUsage(completion_tokens=9, prompt_tokens=79, total_tokens=88, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))
Prompt tokens: 79
Completion tokens: 9
Cached tokens: 0

  📊 USAGE METRICS — Basic Usage Tracking
  Prompt tokens:            79
    ├─ Cached:               0  (❌)
    └─ Uncached:            79
  Completion tokens:         9
  Total tokens:             88
  Cache hit rate:         0.0%
────────────────────────────────────────────────────────────
  💰 COST BREAKDOWN (model: gpt-4o-mini)
  Input (uncached):   $0.000012
  Input (cached):     $0.000000
  Output:             $0.000005
  TOTAL:              $0.000017
  Savings vs no-cache:$0.000000



In [14]:
# ============================================================================
# 3. TECHNIQUE 1: LONG SYSTEM PROMPT — STATIC PREFIX PATTERN
# ============================================================================
"""
THE most common and impactful caching technique.

Strategy: Put a LONG, STATIC system prompt (>1024 tokens) at the start.
Only vary the user message at the end. The system prompt gets cached
after the first call.

PROMPT STRUCTURE:
  ┌──────────────────────────────────────┐
  │ SYSTEM: [long static instructions]   │  ← CACHED after 1st call
  │         [examples, rules, context]   │
  ├──────────────────────────────────────┤
  │ USER: [dynamic per-request query]    │  ← Changes each time
  └──────────────────────────────────────┘
"""

'\nTHE most common and impactful caching technique.\n\nStrategy: Put a LONG, STATIC system prompt (>1024 tokens) at the start.\nOnly vary the user message at the end. The system prompt gets cached\nafter the first call.\n\nPROMPT STRUCTURE:\n  ┌──────────────────────────────────────┐\n  │ SYSTEM: [long static instructions]   │  ← CACHED after 1st call\n  │         [examples, rules, context]   │\n  ├──────────────────────────────────────┤\n  │ USER: [dynamic per-request query]    │  ← Changes each time\n  └──────────────────────────────────────┘\n'

In [44]:
# A long system prompt (needs to be >1024 tokens to trigger caching)
LONG_SYSTEM_PROMPT = """
You are an expert financial analyst assistant. You help users analyze
financial documents, earnings reports, balance sheets, and market data.

## Your Capabilities
- Parse and analyze financial statements (income statement, balance sheet, cash flow)
- Calculate financial ratios (P/E, EV/EBITDA, ROE, ROA, current ratio, etc.)
- Identify trends in revenue, margins, and growth rates
- Compare companies within the same industry
- Assess risk factors and provide balanced analysis

## Output Guidelines
- Always provide specific numbers with sources when available
- Use proper financial terminology
- Flag any assumptions or estimates clearly
- Present both bullish and bearish perspectives
- Include relevant industry benchmarks for comparison

## Financial Ratio Definitions
1. Price-to-Earnings (P/E): Market price per share / Earnings per share
2. Enterprise Value to EBITDA (EV/EBITDA): (Market Cap + Debt - Cash) / EBITDA
3. Return on Equity (ROE): Net Income / Shareholders' Equity
4. Return on Assets (ROA): Net Income / Total Assets
5. Current Ratio: Current Assets / Current Liabilities
6. Quick Ratio: (Current Assets - Inventory) / Current Liabilities
7. Debt-to-Equity (D/E): Total Debt / Shareholders' Equity
8. Gross Margin: (Revenue - COGS) / Revenue
9. Operating Margin: Operating Income / Revenue
10. Net Margin: Net Income / Revenue
11. Free Cash Flow Yield: Free Cash Flow / Market Capitalization
12. Dividend Yield: Annual Dividend Per Share / Price Per Share
13. Price-to-Book (P/B): Market Price Per Share / Book Value Per Share
14. Price-to-Sales (P/S): Market Capitalization / Total Revenue
15. Asset Turnover: Revenue / Average Total Assets

## Industry Classification (GICS)
- Energy (10): Oil, Gas, Consumable Fuels, Energy Equipment & Services
- Materials (15): Chemicals, Construction Materials, Metals & Mining
- Industrials (20): Aerospace, Defense, Machinery, Transportation
- Consumer Discretionary (25): Auto, Retail, Hotels, Media
- Consumer Staples (30): Food, Beverage, Tobacco, Household Products
- Health Care (35): Pharma, Biotech, Health Care Equipment
- Financials (40): Banks, Insurance, Capital Markets, REITs
- Information Technology (45): Software, Hardware, Semiconductors
- Communication Services (50): Telecom, Media, Entertainment
- Utilities (55): Electric, Gas, Water Utilities
- Real Estate (60): REITs, Real Estate Management

## Valuation Methods
1. Discounted Cash Flow (DCF): Project future free cash flows and discount
   them back to present value using WACC.
2. Comparable Company Analysis: Compare valuation multiples (P/E, EV/EBITDA)
   with similar public companies.
3. Precedent Transactions: Look at M&A deal multiples for similar companies.
4. Dividend Discount Model (DDM): For dividend-paying stocks, discount
   future dividends back to present value.
5. Sum of the Parts (SOTP): Value each business segment separately and
   add them together.

## Risk Assessment Framework
- Market Risk: Beta, correlation with market indices
- Credit Risk: Debt ratings, interest coverage ratio
- Liquidity Risk: Bid-ask spread, trading volume
- Operational Risk: Key person dependency, supply chain concentration
- Regulatory Risk: Pending legislation, compliance requirements
- Currency Risk: Foreign revenue exposure, hedging strategies
- Concentration Risk: Customer/supplier concentration

## Common Red Flags in Financial Statements
1. Revenue recognition changes
2. Unusual increase in accounts receivable vs revenue
3. Declining cash flow despite growing earnings
4. Frequent "one-time" charges or adjustments
5. Related party transactions
6. Aggressive capitalization of expenses
7. Significant off-balance-sheet items
8. High executive turnover
9. Delayed or restated financial reports
10. Growing gap between GAAP and non-GAAP earnings

Always maintain objectivity. Present data-driven analysis. Avoid
speculation without clearly labeling it as such.
"""

LONG_SYSTEM_PROMPT = LONG_SYSTEM_PROMPT + (" pad " * 2000)

In [48]:
class FinancialAnalysis(BaseModel):
    """Structured output for financial analysis."""
    company: str = Field(description="Company name or ticker")
    metric: str = Field(description="The financial metric being analyzed")
    value: Optional[float] = Field(None, description="Numeric value if applicable")
    interpretation: str = Field(description="Analysis and interpretation")
    risk_level: str = Field(description="low, medium, or high")


def example_long_system_prompt():
    """
    Demonstrates server-side caching with a long static system prompt.
    Run this function twice quickly — the 2nd call should show cached_tokens > 0.
    """
    model = "gpt-4o-mini"
    questions = [
        "What does a P/E ratio of 45 mean for a SaaS company?",
        "Analyze a company with ROE of 25% and D/E of 2.5",
        "Is a current ratio of 0.8 concerning for a retail company?",
    ]

    for i, question in enumerate(questions):
        result, completion = client.create_with_completion(
            model=model,
            response_model=FinancialAnalysis,
            messages=[
                {"role": "system", "content": LONG_SYSTEM_PROMPT},
                {"role": "user", "content": question},
            ],
            temperature=0.0,
            top_p=1.0,
        )

        print(f"\n📝 Question: {question}")
        print(f"   Answer: {result.company} — {result.interpretation[:80]}...")
        print_usage(completion, label=f"Call {i+1}", model=model)

        # Small delay to let cache propagate (usually instant, but just in case)
        if i == 0:
            time.sleep(1)

In [49]:
example_long_system_prompt()


📝 Question: What does a P/E ratio of 45 mean for a SaaS company?
   Answer: SaaS Company — A P/E ratio of 45 indicates that investors are willing to pay $45 for every $1 o...

  📊 USAGE METRICS — Call 1
  Prompt tokens:         5,019
    ├─ Cached:               0  (❌)
    └─ Uncached:         5,019
  Completion tokens:       113
  Total tokens:          5,132
  Cache hit rate:         0.0%
────────────────────────────────────────────────────────────
  💰 COST BREAKDOWN (model: gpt-4o-mini)
  Input (uncached):   $0.000753
  Input (cached):     $0.000000
  Output:             $0.000068
  TOTAL:              $0.000821
  Savings vs no-cache:$0.000000


📝 Question: Analyze a company with ROE of 25% and D/E of 2.5
   Answer: Company with ROE of 25% and D/E of 2.5 — A Return on Equity (ROE) of 25% indicates that the company is generating a stron...

  📊 USAGE METRICS — Call 2
  Prompt tokens:         5,021
    ├─ Cached:           4,864  (✅)
    └─ Uncached:           157
  Completion tokens

In [24]:
# ============================================================================
# 4. TECHNIQUE 2: TOOL/FUNCTION DEFINITIONS CACHING
# ============================================================================
"""
Tool definitions are part of the prompt prefix. If you have MANY tools
defined identically across requests, they'll be cached.

KEY: Tool definitions and their ORDER must be IDENTICAL between requests.
     Changing the order = cache miss.

PROMPT STRUCTURE:
  ┌──────────────────────────────────────┐
  │ TOOLS: [tool_1, tool_2, ... tool_n]  │  ← CACHED (keep order fixed!)
  │ SYSTEM: [instructions]               │  ← CACHED
  ├──────────────────────────────────────┤
  │ USER: [dynamic query]                │  ← Changes each time
  └──────────────────────────────────────┘
"""

"\nTool definitions are part of the prompt prefix. If you have MANY tools\ndefined identically across requests, they'll be cached.\n\nKEY: Tool definitions and their ORDER must be IDENTICAL between requests.\n     Changing the order = cache miss.\n\nPROMPT STRUCTURE:\n  ┌──────────────────────────────────────┐\n  │ TOOLS: [tool_1, tool_2, ... tool_n]  │  ← CACHED (keep order fixed!)\n  │ SYSTEM: [instructions]               │  ← CACHED\n  ├──────────────────────────────────────┤\n  │ USER: [dynamic query]                │  ← Changes each time\n  └──────────────────────────────────────┘\n"

In [25]:
# Many tool definitions (contributes to the >1024 token threshold)
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "Get the current stock price for a given ticker symbol. Returns real-time market data including bid, ask, last price, and volume.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {"type": "string", "description": "Stock ticker symbol (e.g., AAPL, MSFT)"},
                    "exchange": {"type": "string", "description": "Stock exchange (e.g., NYSE, NASDAQ)"},
                },
                "required": ["ticker"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_financial_statements",
            "description": "Retrieve financial statements for a company including income statement, balance sheet, and cash flow statement for a specified period.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {"type": "string", "description": "Stock ticker symbol"},
                    "statement_type": {
                        "type": "string",
                        "enum": ["income_statement", "balance_sheet", "cash_flow"],
                        "description": "Type of financial statement",
                    },
                    "period": {
                        "type": "string",
                        "enum": ["annual", "quarterly"],
                        "description": "Reporting period",
                    },
                    "years": {"type": "integer", "description": "Number of years of data"},
                },
                "required": ["ticker", "statement_type"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "compare_companies",
            "description": "Compare financial metrics across multiple companies in the same industry or sector.",
            "parameters": {
                "type": "object",
                "properties": {
                    "tickers": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of ticker symbols to compare",
                    },
                    "metrics": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Financial metrics to compare",
                    },
                },
                "required": ["tickers", "metrics"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_analyst_ratings",
            "description": "Get analyst ratings, price targets, and consensus recommendations for a stock.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {"type": "string", "description": "Stock ticker symbol"},
                    "include_price_targets": {"type": "boolean", "description": "Include price target data"},
                },
                "required": ["ticker"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "screen_stocks",
            "description": "Screen stocks based on financial criteria and filters.",
            "parameters": {
                "type": "object",
                "properties": {
                    "min_market_cap": {"type": "number", "description": "Minimum market cap in billions"},
                    "max_pe_ratio": {"type": "number", "description": "Maximum P/E ratio"},
                    "min_dividend_yield": {"type": "number", "description": "Minimum dividend yield percentage"},
                    "sector": {"type": "string", "description": "GICS sector filter"},
                    "min_revenue_growth": {"type": "number", "description": "Minimum YoY revenue growth %"},
                },
                "required": [],
            },
        },
    },
]

In [50]:
def example_tool_caching():
    """
    Demonstrates caching of tool definitions.
    Same tools + same system prompt = cached prefix for subsequent calls.
    """
    model = "gpt-4o-mini"
    system_msg = (
        "You are a financial assistant with access to market data tools. "
        "Use the available tools to help users with their queries. "
        "Always explain your reasoning before calling a tool."
    )

    queries = [
        "What's Apple's current stock price?",
        "Compare AAPL and MSFT on P/E ratio and revenue growth",
    ]

    for i, query in enumerate(queries):
        # Using raw OpenAI client here to show tool usage
        response = raw_client.chat.completions.create(
            model=model,
            tools=TOOLS,
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": query},
            ],
            temperature=0.0,
        )

        print(f"\n🔧 Query: {query}")
        if response.choices[0].message.tool_calls:
            for tc in response.choices[0].message.tool_calls:
                print(f"   Tool called: {tc.function.name}({tc.function.arguments})")
        print_usage(response, label=f"Tool Call {i+1}", model=model)

        if i == 0:
            time.sleep(1)

In [51]:
example_tool_caching()


🔧 Query: What's Apple's current stock price?
   Tool called: get_stock_price({"ticker":"AAPL","exchange":"NASDAQ"})

  📊 USAGE METRICS — Tool Call 1
  Prompt tokens:           396
    ├─ Cached:               0  (❌)
    └─ Uncached:           396
  Completion tokens:        20
  Total tokens:            416
  Cache hit rate:         0.0%
────────────────────────────────────────────────────────────
  💰 COST BREAKDOWN (model: gpt-4o-mini)
  Input (uncached):   $0.000059
  Input (cached):     $0.000000
  Output:             $0.000012
  TOTAL:              $0.000071
  Savings vs no-cache:$0.000000


🔧 Query: Compare AAPL and MSFT on P/E ratio and revenue growth
   Tool called: compare_companies({"tickers": ["AAPL", "MSFT"], "metrics": ["P/E ratio"]})
   Tool called: compare_companies({"tickers": ["AAPL", "MSFT"], "metrics": ["revenue growth"]})

  📊 USAGE METRICS — Tool Call 2
  Prompt tokens:           403
    ├─ Cached:               0  (❌)
    └─ Uncached:           403
  Completion to

In [52]:
# ============================================================================
# 5. TECHNIQUE 3: STRUCTURED OUTPUT SCHEMA CACHING
# ============================================================================
"""
When using Instructor, the Pydantic model schema becomes part of the
prompt prefix (as a tool definition or structured output schema).
This means the schema itself gets cached!

KEY INSIGHT: The schema serves as a prefix to the system message.
             Keep your Pydantic models STABLE between calls.

PROMPT STRUCTURE:
  ┌──────────────────────────────────────┐
  │ SCHEMA: [Pydantic model JSON schema] │  ← CACHED
  │ SYSTEM: [instructions]               │  ← CACHED
  ├──────────────────────────────────────┤
  │ USER: [dynamic query]                │  ← Changes each time
  └──────────────────────────────────────┘
"""

'\nWhen using Instructor, the Pydantic model schema becomes part of the\nprompt prefix (as a tool definition or structured output schema).\nThis means the schema itself gets cached!\n\nKEY INSIGHT: The schema serves as a prefix to the system message.\n             Keep your Pydantic models STABLE between calls.\n\nPROMPT STRUCTURE:\n  ┌──────────────────────────────────────┐\n  │ SCHEMA: [Pydantic model JSON schema] │  ← CACHED\n  │ SYSTEM: [instructions]               │  ← CACHED\n  ├──────────────────────────────────────┤\n  │ USER: [dynamic query]                │  ← Changes each time\n  └──────────────────────────────────────┘\n'

In [53]:
class DetailedCompanyProfile(BaseModel):
    """A complex schema that contributes significant tokens to the prefix."""
    name: str = Field(description="Full legal company name")
    ticker: str = Field(description="Stock ticker symbol")
    sector: str = Field(description="GICS sector classification")
    industry: str = Field(description="Specific industry within sector")
    market_cap_billions: float = Field(description="Market capitalization in USD billions")
    pe_ratio: Optional[float] = Field(None, description="Trailing twelve month P/E ratio")
    forward_pe: Optional[float] = Field(None, description="Forward P/E based on next year estimates")
    revenue_growth_pct: Optional[float] = Field(None, description="Year-over-year revenue growth percentage")
    profit_margin_pct: Optional[float] = Field(None, description="Net profit margin percentage")
    debt_to_equity: Optional[float] = Field(None, description="Total debt to equity ratio")
    dividend_yield_pct: Optional[float] = Field(None, description="Annual dividend yield percentage")
    analyst_consensus: str = Field(description="buy, hold, or sell consensus")
    key_risks: list[str] = Field(description="Top 3-5 risk factors")
    competitive_advantages: list[str] = Field(description="Key competitive moats")
    summary: str = Field(description="2-3 sentence investment summary")


def example_schema_caching():
    """
    Demonstrates that the structured output schema is cached.
    Multiple extractions using the same schema benefit from caching.
    """
    model = "gpt-4o-mini"
    companies = [
        "Describe Apple Inc as a public company with estimated financials",
        "Describe Microsoft Corporation as a public company with estimated financials",
        "Describe Tesla Inc as a public company with estimated financials",
    ]

    for i, prompt in enumerate(companies):
        result, completion = client.create_with_completion(
            model=model,
            response_model=DetailedCompanyProfile,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a financial data extraction assistant. "
                        "Extract company profiles with estimated current financial data. "
                        "Use your best knowledge for approximate figures. "
                        "Be specific with numbers."
                    ),
                },
                {"role": "user", "content": prompt},
            ],
            temperature=0.0,
        )

        print(f"\n🏢 {result.name} ({result.ticker})")
        print(f"   Sector: {result.sector} | P/E: {result.pe_ratio}")
        print(f"   Summary: {result.summary[:80]}...")
        print_usage(completion, label=f"Schema Cache — Call {i+1}", model=model)

        if i == 0:
            time.sleep(1)

In [54]:
example_schema_caching()


🏢 Apple Inc. (AAPL)
   Sector: Information Technology | P/E: 28.0
   Summary: Apple Inc. is a leading technology company known for its innovative consumer ele...

  📊 USAGE METRICS — Schema Cache — Call 1
  Prompt tokens:           393
    ├─ Cached:               0  (❌)
    └─ Uncached:           393
  Completion tokens:       162
  Total tokens:            555
  Cache hit rate:         0.0%
────────────────────────────────────────────────────────────
  💰 COST BREAKDOWN (model: gpt-4o-mini)
  Input (uncached):   $0.000059
  Input (cached):     $0.000000
  Output:             $0.000097
  TOTAL:              $0.000156
  Savings vs no-cache:$0.000000


🏢 Microsoft Corporation (MSFT)
   Sector: Information Technology | P/E: 34.0
   Summary: Microsoft Corporation is a leading technology company known for its software pro...

  📊 USAGE METRICS — Schema Cache — Call 2
  Prompt tokens:           393
    ├─ Cached:               0  (❌)
    └─ Uncached:           393
  Completion tokens:      

In [55]:
# ============================================================================
# 7. TECHNIQUE 5: LARGE CONTEXT DOCUMENT CACHING
# ============================================================================
"""
When you include a large document in your prompt and ask multiple
questions about it, the document becomes a cached prefix.

PROMPT STRUCTURE:
  ┌──────────────────────────────────────┐
  │ SYSTEM: [instructions]               │  ← CACHED
  │ USER: "<document>                    │
  │   [large document text, 5K+ tokens]  │  ← CACHED
  │ </document>                          │
  │                                      │
  │ Now answer: [specific question]"     │  ← Changes each time
  └──────────────────────────────────────┘

NOTE: The document MUST be in the same position and identical every time.
"""

SAMPLE_DOCUMENT = """
# Annual Report 2024 — TechCorp Industries

## Executive Summary
TechCorp Industries reported record revenue of $45.2 billion for fiscal year 2024,
representing a 23% increase year-over-year. The company's cloud computing division
drove the majority of growth, with cloud revenue reaching $18.7 billion (+34% YoY).
Operating income improved to $12.1 billion, with operating margins expanding to 26.8%
from 24.2% in the prior year. The company generated $15.3 billion in free cash flow
and returned $8.2 billion to shareholders through dividends and share repurchases.

## Revenue Breakdown by Segment
### Cloud Computing Division ($18.7B, +34% YoY)
The cloud computing division continued its strong growth trajectory, driven by
enterprise adoption of AI and machine learning workloads. Key metrics include:
- Infrastructure as a Service (IaaS): $8.2B (+41% YoY)
- Platform as a Service (PaaS): $5.8B (+32% YoY)
- Software as a Service (SaaS): $4.7B (+24% YoY)
- Annual recurring revenue (ARR) reached $21.3B
- Net revenue retention rate: 128%
- Number of $1M+ ARR customers: 2,847 (+18%)

### Enterprise Software Division ($15.3B, +12% YoY)
The enterprise software segment showed steady growth with increasing subscription
mix. Subscription revenue now represents 78% of segment revenue, up from 71%.
- Database products: $6.1B (+8% YoY)
- Enterprise applications: $5.4B (+15% YoY)
- Development tools: $3.8B (+14% YoY)

### Hardware Division ($8.4B, +18% YoY)
Hardware revenue benefited from strong demand for AI-optimized server configurations.
- Server systems: $4.2B (+28% YoY)
- Networking equipment: $2.8B (+12% YoY)
- Storage solutions: $1.4B (+5% YoY)

### Professional Services ($2.8B, +9% YoY)
Consulting and implementation services grew moderately as automation reduced
some service engagement sizes while increasing overall volume.

## Profitability Analysis
- Gross margin: 67.2% (up from 65.8%)
- Operating margin: 26.8% (up from 24.2%)
- Net margin: 22.1% (up from 19.8%)
- Adjusted EBITDA margin: 34.5%
- R&D expense: $8.9B (19.7% of revenue)
- Sales & marketing: $6.2B (13.7% of revenue)
- G&A expense: $3.1B (6.9% of revenue)

## Balance Sheet Highlights
- Total assets: $98.7B
- Cash and equivalents: $22.4B
- Total debt: $18.9B
- Shareholders' equity: $52.3B
- Current ratio: 2.1x
- Debt-to-equity ratio: 0.36x

## Cash Flow Statement
- Operating cash flow: $18.2B
- Capital expenditures: ($2.9B)
- Free cash flow: $15.3B
- Dividends paid: ($3.1B)
- Share repurchases: ($5.1B)
- Acquisitions: ($2.3B)

## Guidance for FY2025
- Revenue: $52-54 billion (15-19% growth)
- Operating margin: 27-28%
- Cloud revenue: $24-26 billion (28-39% growth)
- Capital expenditures: $3.5-4.0 billion
- Free cash flow: $16-18 billion
- Effective tax rate: 19-21%
"""

In [56]:
SAMPLE_DOCUMENT = SAMPLE_DOCUMENT + (" Test Data " * 2000)

In [57]:
class DocumentInsight(BaseModel):
    question: str
    answer: str
    supporting_data: list[str] = Field(description="Key data points that support the answer")
    confidence: str = Field(description="high, medium, or low")


def example_document_caching():
    """
    Ask multiple questions about the same document.
    The document becomes a cached prefix after the first call.
    """
    model = "gpt-4o-mini"

    questions = [
        "What was TechCorp's total revenue and YoY growth?",
        "Which segment had the highest growth rate?",
        "What is the company's debt-to-equity ratio and is it healthy?",
        "What is management guiding for FY2025 cloud revenue?",
    ]

    for i, question in enumerate(questions):
        result, completion = client.create_with_completion(
            model=model,
            response_model=DocumentInsight,
            messages=[
                {
                    "role": "system",
                    "content": "You analyze financial documents and extract precise answers with supporting data.",
                },
                {
                    "role": "user",
                    "content": f"""<document>
{SAMPLE_DOCUMENT}
</document>

Based on the document above, answer this question: {question}""",
                },
            ],
        )

        print(f"\n📄 Q: {question}")
        print(f"   A: {result.answer}")
        print(f"   Data: {result.supporting_data}")
        print_usage(completion, label=f"Document Q&A — Call {i+1}", model=model)

        if i == 0:
            time.sleep(1)


In [58]:
example_document_caching()


📄 Q: What was TechCorp's total revenue and YoY growth?
   A: TechCorp's total revenue for fiscal year 2024 was $45.2 billion, representing a 23% increase year-over-year.
   Data: ['Total revenue: $45.2 billion', 'Year-over-year growth: 23%']

  📊 USAGE METRICS — Document Q&A — Call 1
  Prompt tokens:         6,971
    ├─ Cached:               0  (❌)
    └─ Uncached:         6,971
  Completion tokens:        73
  Total tokens:          7,044
  Cache hit rate:         0.0%
────────────────────────────────────────────────────────────
  💰 COST BREAKDOWN (model: gpt-4o-mini)
  Input (uncached):   $0.001046
  Input (cached):     $0.000000
  Output:             $0.000044
  TOTAL:              $0.001089
  Savings vs no-cache:$0.000000


📄 Q: Which segment had the highest growth rate?
   A: The Cloud Computing Division had the highest growth rate at 34% year-over-year.
   Data: ['Cloud Computing Division: $18.7B (+34% YoY)', 'Enterprise Software Division: $15.3B (+12% YoY)', 'Hardware Division

In [59]:
# ============================================================================
# 8. TECHNIQUE 6: prompt_cache_key FOR ROUTING HINTS
# ============================================================================
"""
OpenAI routes requests based on a hash of the initial ~256 tokens of
the prompt. When many requests share the same long prefix, you can use
`prompt_cache_key` to help OpenAI route them to the same server.

This is a ROUTING HINT, not a cache breakpoint. It helps improve cache
hit rates for high-throughput applications.

WHEN TO USE:
- You have many concurrent requests with similar prefixes
- You want to group requests by tenant/user/session
- Your cache hit rate is lower than expected

LIMITS:
- Keep each unique prefix + prompt_cache_key combo under 15 req/min
  to avoid cache overflow
"""

'\nOpenAI routes requests based on a hash of the initial ~256 tokens of\nthe prompt. When many requests share the same long prefix, you can use\n`prompt_cache_key` to help OpenAI route them to the same server.\n\nThis is a ROUTING HINT, not a cache breakpoint. It helps improve cache\nhit rates for high-throughput applications.\n\nWHEN TO USE:\n- You have many concurrent requests with similar prefixes\n- You want to group requests by tenant/user/session\n- Your cache hit rate is lower than expected\n\nLIMITS:\n- Keep each unique prefix + prompt_cache_key combo under 15 req/min\n  to avoid cache overflow\n'

In [60]:
def example_prompt_cache_key():
    """
    Demonstrates using prompt_cache_key for better routing.
    """
    model = "gpt-4o-mini"

    # Different user queries, same system prompt
    # Use a consistent cache key for all requests from the same "tenant"
    for i, query in enumerate(["Explain P/E ratio", "Explain EV/EBITDA"]):
        response = raw_client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": LONG_SYSTEM_PROMPT},
                {"role": "user", "content": query},
            ],
            # Extra body params (if supported by your SDK version)
            # Note: prompt_cache_key may need to be passed via extra_body
            extra_body={"prompt_cache_key": "tenant-12345"},
            temperature=0.0,
        )

        print(f"\n🔑 Query with cache key: {query}")
        print_usage(response, label=f"Cache Key — Call {i+1}", model=model)

        if i == 0:
            time.sleep(1)

In [61]:
example_prompt_cache_key()


🔑 Query with cache key: Explain P/E ratio

  📊 USAGE METRICS — Cache Key — Call 1
  Prompt tokens:         4,894
    ├─ Cached:               0  (❌)
    └─ Uncached:         4,894
  Completion tokens:       485
  Total tokens:          5,379
  Cache hit rate:         0.0%
────────────────────────────────────────────────────────────
  💰 COST BREAKDOWN (model: gpt-4o-mini)
  Input (uncached):   $0.000734
  Input (cached):     $0.000000
  Output:             $0.000291
  TOTAL:              $0.001025
  Savings vs no-cache:$0.000000


🔑 Query with cache key: Explain EV/EBITDA

  📊 USAGE METRICS — Cache Key — Call 2
  Prompt tokens:         4,896
    ├─ Cached:           4,224  (✅)
    └─ Uncached:           672
  Completion tokens:       547
  Total tokens:          5,443
  Cache hit rate:        86.3%
────────────────────────────────────────────────────────────
  💰 COST BREAKDOWN (model: gpt-4o-mini)
  Input (uncached):   $0.000101
  Input (cached):     $0.000317
  Output:             $0.

In [62]:
# ============================================================================
# 9. TECHNIQUE 7: INSTRUCTOR'S BUILT-IN CLIENT-SIDE CACHE (AutoCache)
# ============================================================================
"""
Starting with Instructor v1.9.1, you can enable client-side caching
that stores responses locally. This is DIFFERENT from OpenAI's
server-side caching:

  SERVER-SIDE (OpenAI):   Caches the KV computation → 50% cheaper tokens
  CLIENT-SIDE (Instructor): Caches the full response → 0 API calls on hit

Instructor supports:
  - AutoCache: In-memory LRU cache
  - DiskCache: Persistent cache on disk

The cache key is a SHA-256 hash of: model + messages + mode + response_model schema.
Any change to the Pydantic model (field names, types, descriptions) auto-busts the cache.
"""

"\nStarting with Instructor v1.9.1, you can enable client-side caching\nthat stores responses locally. This is DIFFERENT from OpenAI's\nserver-side caching:\n\n  SERVER-SIDE (OpenAI):   Caches the KV computation → 50% cheaper tokens\n  CLIENT-SIDE (Instructor): Caches the full response → 0 API calls on hit\n\nInstructor supports:\n  - AutoCache: In-memory LRU cache\n  - DiskCache: Persistent cache on disk\n\nThe cache key is a SHA-256 hash of: model + messages + mode + response_model schema.\nAny change to the Pydantic model (field names, types, descriptions) auto-busts the cache.\n"

In [63]:
from instructor.cache import AutoCache  # , DiskCache  # uncomment if using disk


def example_instructor_autocache():
    """
    Demonstrates Instructor's built-in AutoCache — zero API calls on repeat.
    """
    # Create a client with AutoCache enabled
    cached_client = instructor.from_provider(
        "openai/gpt-4o-mini",
        cache=AutoCache(maxsize=100),  # LRU cache, max 100 entries
    )

    class ExtractedEntity(BaseModel):
        name: str
        entity_type: str  # person, company, location, etc.
        context: str

    prompt = "Extract the main entity: Elon Musk announced new Tesla factory in Berlin."

    # Call 1: Cache miss — makes real API call
    start = time.perf_counter()
    result1 = cached_client.create(
        response_model=ExtractedEntity,
        messages=[{"role": "user", "content": prompt}],
    )
    time1 = time.perf_counter() - start

    # Call 2: Cache hit — no API call, instant response
    start = time.perf_counter()
    result2 = cached_client.create(
        response_model=ExtractedEntity,
        messages=[{"role": "user", "content": prompt}],
    )
    time2 = time.perf_counter() - start

    print(f"\n🗄️  AutoCache Demo")
    print(f"   Call 1 (miss): {time1:.4f}s → {result1}")
    print(f"   Call 2 (hit):  {time2:.6f}s → {result2}")
    print(f"   Speedup: {time1/time2:.0f}x faster")
    print(f"   Results identical: {result1 == result2}")

In [64]:
example_instructor_autocache()


🗄️  AutoCache Demo
   Call 1 (miss): 1.0443s → name='Elon Musk' entity_type='Person' context='Elon Musk announced new Tesla factory in Berlin.'
   Call 2 (hit):  0.001986s → name='Elon Musk' entity_type='Person' context='Elon Musk announced new Tesla factory in Berlin.'
   Speedup: 526x faster
   Results identical: False


In [65]:
# ============================================================================
# 10. TECHNIQUE 8: functools.cache — IN-MEMORY CLIENT-SIDE CACHE
# ============================================================================
"""
The simplest caching approach: Python's built-in functools.cache.
Caches function results in memory based on arguments.

PROS: Zero dependencies, dead simple
CONS: Not persistent (lost when process exits), memory-bound

BEST FOR: Scripts, notebooks, or services where you call the same
          extraction multiple times in a single run.
"""

"\nThe simplest caching approach: Python's built-in functools.cache.\nCaches function results in memory based on arguments.\n\nPROS: Zero dependencies, dead simple\nCONS: Not persistent (lost when process exits), memory-bound\n\nBEST FOR: Scripts, notebooks, or services where you call the same\n          extraction multiple times in a single run.\n"

In [66]:
import functools


@functools.cache
def extract_user(data: str) -> UserInfo:
    """Cached extraction — same input always returns cached result."""
    return client.create(
        response_model=UserInfo,
        messages=[{"role": "user", "content": data}],
    )


def example_functools_cache():
    """
    Demonstrates functools.cache for simple in-memory caching.
    """
    # Call 1: Cache miss
    start = time.perf_counter()
    user1 = extract_user("Extract: Maria is 30 years old")
    t1 = time.perf_counter() - start

    # Call 2: Cache hit (exact same argument)
    start = time.perf_counter()
    user2 = extract_user("Extract: Maria is 30 years old")
    t2 = time.perf_counter() - start

    # Call 3: Cache miss (different argument)
    start = time.perf_counter()
    user3 = extract_user("Extract: John is 28 years old")
    t3 = time.perf_counter() - start

    print(f"\n⚡ functools.cache Demo")
    print(f"   Call 1 (miss):     {t1:.4f}s → {user1}")
    print(f"   Call 2 (HIT):      {t2:.8f}s → {user2}")
    print(f"   Call 3 (miss/new): {t3:.4f}s → {user3}")
    print(f"   Call 2 speedup: {t1/t2:,.0f}x")

In [67]:
example_functools_cache()


⚡ functools.cache Demo
   Call 1 (miss):     0.7968s → name='Maria' age=30
   Call 2 (HIT):      0.00000046s → name='Maria' age=30
   Call 3 (miss/new): 0.6558s → name='John' age=28
   Call 2 speedup: 1,739,811x


In [68]:
# ============================================================================
# 11. TECHNIQUE 9: diskcache — PERSISTENT CLIENT-SIDE CACHE
# ============================================================================
"""
diskcache persists cached results to disk. Survives process restarts.

pip install diskcache

BEST FOR: Long-running data pipelines, batch processing, development
          where you don't want to re-call the API on restart.
"""

"\ndiskcache persists cached results to disk. Survives process restarts.\n\npip install diskcache\n\nBEST FOR: Long-running data pipelines, batch processing, development\n          where you don't want to re-call the API on restart.\n"

In [69]:
import inspect


def make_diskcache_decorator(cache_dir="./instructor_cache"):
    """Creates a caching decorator that persists to disk."""
    try:
        import diskcache
        cache = diskcache.Cache(cache_dir)
    except ImportError:
        print("⚠️  Install diskcache: pip install diskcache")
        return lambda f: f  # No-op if not installed

    def decorator(func):
        return_type = inspect.signature(func).return_annotation
        if not issubclass(return_type, BaseModel):
            raise ValueError("Return type must be a Pydantic BaseModel")

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            key = f"{func.__name__}-{functools._make_key(args, kwargs, typed=False)}"

            # Check disk cache
            cached = cache.get(key)
            if cached is not None:
                return return_type.model_validate_json(cached)

            # Cache miss — call API
            result = func(*args, **kwargs)
            cache.set(key, result.model_dump_json())
            return result

        wrapper.cache = cache  # Expose cache for management
        return wrapper

    return decorator


disk_cache = make_diskcache_decorator("./my_llm_cache")


@disk_cache
def extract_company(description: str) -> DetailedCompanyProfile:
    """Disk-cached company extraction."""
    return client.create(
        model="gpt-4o-mini",
        response_model=DetailedCompanyProfile,
        messages=[
            {
                "role": "system",
                "content": "Extract a detailed company profile from the description.",
            },
            {"role": "user", "content": description},
        ],
    )


def example_diskcache():
    """
    Demonstrates persistent disk caching.
    """
    desc = "Apple Inc is a tech giant trading at AAPL with roughly $3T market cap"

    start = time.perf_counter()
    profile1 = extract_company(desc)
    t1 = time.perf_counter() - start

    start = time.perf_counter()
    profile2 = extract_company(desc)
    t2 = time.perf_counter() - start

    print(f"\n💾 Disk Cache Demo")
    print(f"   Call 1: {t1:.4f}s → {profile1.name}")
    print(f"   Call 2: {t2:.6f}s → {profile2.name} (from disk)")
    print(f"   Speedup: {t1/max(t2, 0.000001):,.0f}x")


In [70]:
example_diskcache()


💾 Disk Cache Demo
   Call 1: 2.4537s → Apple Inc
   Call 2: 0.000136s → Apple Inc (from disk)
   Speedup: 18,020x


In [71]:
# ============================================================================
# 12. TECHNIQUE 10: REDIS — DISTRIBUTED CLIENT-SIDE CACHE
# ============================================================================
"""
Redis caching for distributed/multi-process applications.

pip install redis

BEST FOR: Production microservices, multi-worker setups, shared cache
          across different application instances.
"""

'\nRedis caching for distributed/multi-process applications.\n\npip install redis\n\nBEST FOR: Production microservices, multi-worker setups, shared cache\n          across different application instances.\n'

In [72]:
def make_redis_decorator(host="localhost", port=6379, ttl_seconds=3600):
    """Creates a Redis-backed caching decorator."""
    try:
        import redis
        cache = redis.Redis(host=host, port=port)
        cache.ping()  # Test connection
    except Exception:
        print("⚠️  Redis not available. Skipping Redis cache.")
        return lambda f: f

    def decorator(func):
        return_type = inspect.signature(func).return_annotation
        if not issubclass(return_type, BaseModel):
            raise ValueError("Return type must be a Pydantic BaseModel")

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            key = f"{func.__name__}-{functools._make_key(args, kwargs, typed=False)}"

            cached = cache.get(key)
            if cached is not None:
                return return_type.model_validate_json(cached)

            result = func(*args, **kwargs)
            cache.set(key, result.model_dump_json(), ex=ttl_seconds)
            return result

        return wrapper

    return decorator

In [73]:
# ============================================================================
# 13. COST CALCULATOR UTILITY
# ============================================================================

def estimate_cost(
    model: str,
    prompt_tokens: int,
    completion_tokens: int,
    cached_tokens: int = 0,
) -> dict:
    """
    Estimate cost for an API call with optional caching.

    Returns:
        dict with cost breakdown and savings info
    """
    pricing = PRICING.get(model, PRICING["gpt-4o-mini"])
    uncached = prompt_tokens - cached_tokens

    actual_cost = (
        (uncached / 1_000_000) * pricing["input"]
        + (cached_tokens / 1_000_000) * pricing["cached_input"]
        + (completion_tokens / 1_000_000) * pricing["output"]
    )

    no_cache_cost = (
        (prompt_tokens / 1_000_000) * pricing["input"]
        + (completion_tokens / 1_000_000) * pricing["output"]
    )

    return {
        "model": model,
        "actual_cost": round(actual_cost, 8),
        "without_cache_cost": round(no_cache_cost, 8),
        "savings": round(no_cache_cost - actual_cost, 8),
        "savings_pct": round((1 - actual_cost / no_cache_cost) * 100, 1) if no_cache_cost > 0 else 0,
    }


def example_cost_estimation():
    """Show cost estimation for different scenarios."""
    scenarios = [
        ("gpt-4o-mini", 5000, 200, 0, "No caching"),
        ("gpt-4o-mini", 5000, 200, 4000, "80% cached"),
        ("gpt-4o", 5000, 200, 0, "No caching"),
        ("gpt-4o", 5000, 200, 4000, "80% cached"),
        ("gpt-4.1", 5000, 200, 0, "No caching"),
        ("gpt-4.1", 5000, 200, 4500, "90% cached"),
    ]

    print("\n" + "=" * 75)
    print("  💰 COST COMPARISON TABLE")
    print("=" * 75)
    print(f"  {'Model':<15} {'Scenario':<15} {'Actual':<12} {'No-Cache':<12} {'Savings':<10} {'%':>5}")
    print("  " + "-" * 69)

    for model, pt, ct, cached, label in scenarios:
        result = estimate_cost(model, pt, ct, cached)
        print(
            f"  {model:<15} {label:<15} "
            f"${result['actual_cost']:<10.6f} "
            f"${result['without_cache_cost']:<10.6f} "
            f"${result['savings']:<8.6f} "
            f"{result['savings_pct']:>5.1f}%"
        )

In [74]:
# ============================================================================
# 14. BEST PRACTICES CHEAT SHEET
# ============================================================================

BEST_PRACTICES = """
╔══════════════════════════════════════════════════════════════════════════╗
║                    PROMPT CACHING BEST PRACTICES                       ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                        ║
║  🏗️  PROMPT STRUCTURE (most important!)                                ║
║  ──────────────────────────────────────                                ║
║  1. Static content FIRST → system prompt, tool defs, schemas,          ║
║     examples, reference documents                                      ║
║  2. Dynamic content LAST → user query, variable context                ║
║  3. Never reorder/modify earlier messages in conversation              ║
║                                                                        ║
║  📏  TOKEN THRESHOLDS                                                  ║
║  ──────────────────────                                                ║
║  • Minimum 1,024 tokens to trigger caching                            ║
║  • Cached in 128-token increments after 1,024                          ║
║  • Longer stable prefixes = more savings                               ║
║                                                                        ║
║  ⏰  CACHE LIFETIME                                                    ║
║  ─────────────────                                                     ║
║  • Default: 5-10 minutes of inactivity                                 ║
║  • gpt-4.1 series: 24-hour retention available                         ║
║  • Keep requests flowing to avoid eviction                             ║
║                                                                        ║
║  🔧  TOOL DEFINITIONS                                                  ║
║  ─────────────────                                                     ║
║  • Keep tool order IDENTICAL across requests                           ║
║  • Don't add/remove tools between calls if possible                    ║
║  • Tool definitions count toward the cached prefix                     ║
║                                                                        ║
║  📊  MONITORING                                                        ║
║  ────────────                                                          ║
║  • Check `usage.prompt_tokens_details.cached_tokens` every call        ║
║  • Use `create_with_completion()` in Instructor for full metrics       ║
║  • Track cache hit rate over time                                      ║
║                                                                        ║
║  💡  INSTRUCTOR-SPECIFIC TIPS                                          ║
║  ────────────────────────────                                          ║
║  • Stable Pydantic models = stable schema prefix = better caching      ║
║  • Combine server-side (OpenAI) + client-side (AutoCache) caching      ║
║  • Use create_with_completion() to always see usage stats              ║
║                                                                        ║
║  🚫  WHAT BREAKS CACHING                                              ║
║  ─────────────────────                                                 ║
║  • Changing any token in the prefix (even whitespace)                  ║
║  • Reordering messages or tool definitions                             ║
║  • Dynamic content (timestamps, random IDs) in system prompt           ║
║  • Different model versions (gpt-4o vs gpt-4o-mini)                   ║
║  • Summarizing/compacting conversation history mid-chat                ║
║                                                                        ║
╚══════════════════════════════════════════════════════════════════════════╝
"""


# ============================================================================
# MAIN — Run all examples
# ============================================================================

def main():
    print(BEST_PRACTICES)
    print("\n🚀 Running Prompt Caching Examples...\n")

    # Uncomment the examples you want to run:

    # --- Server-side caching (requires API key) ---
    # example_track_usage()
    # example_long_system_prompt()
    # example_tool_caching()
    # example_schema_caching()
    # example_multi_turn_caching()
    # example_document_caching()
    # example_prompt_cache_key()

    # --- Client-side caching (requires API key for first call) ---
    # example_instructor_autocache()
    # example_functools_cache()
    # example_diskcache()

    # --- No API key needed ---
    example_cost_estimation()

    print("\n✅ Done! Uncomment other examples in main() to run them.")
    print("   Remember: You need OPENAI_API_KEY set for API-calling examples.")


if __name__ == "__main__":
    main()


╔══════════════════════════════════════════════════════════════════════════╗
║                    PROMPT CACHING BEST PRACTICES                       ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                        ║
║  🏗️  PROMPT STRUCTURE (most important!)                                ║
║  ──────────────────────────────────────                                ║
║  1. Static content FIRST → system prompt, tool defs, schemas,          ║
║     examples, reference documents                                      ║
║  2. Dynamic content LAST → user query, variable context                ║
║  3. Never reorder/modify earlier messages in conversation              ║
║                                                                        ║
║  📏  TOKEN THRESHOLDS                                                  ║
║  ──────────────────────                                                ║
║  • Minimum 1,024 to